In [ ]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from geopy.distance import geodesic

# --- STEP 1: Load Data ---

# Ganti path dengan file aslinya
excel_file = "1742574481558_Dataset-ME-2025.xlsx"

customer_df = pd.read_excel(excel_file, sheet_name="Customer Code", skiprows=0)
dc_df = pd.read_excel(excel_file, sheet_name="DC Final", skiprows=0)

# Pilih area tertentu untuk contoh (misal: Bali Nusra)
area_filter = 'Kalimantan'
filtered_customers = customer_df[customer_df['Sub-District Area'].str.contains(area_filter, na=False)]

# --- STEP 2: Persiapkan koordinat & demand ---

# Ambil data koordinat & total permintaan pelanggan
locations = list(zip(filtered_customers['Latitude'], filtered_customers['Longitude']))
demands = filtered_customers.iloc[:, 5:].sum(axis=1).fillna(0).astype(int).tolist()  # Total demand per pelanggan

# Ambil DC terdekat (misalnya: satu DC dulu sebagai depot)
dc_location = dc_df[dc_df['Distributor Area'].str.contains(area_filter, na=False)].iloc[0]
depot_coord = (dc_location['Latitude'], dc_location['Longitude'])

# Gabungkan depot + pelanggan
all_locations = [depot_coord] + locations
all_demands = [0] + demands  # Depot tidak punya demand

# --- STEP 3: Bangun Distance Matrix ---
def compute_distance_matrix(locations):
    distance_matrix = []
    for from_node in locations:
        row = []
        for to_node in locations:
            distance = geodesic(from_node, to_node).km
            row.append(int(distance))  # Integer untuk OR-Tools
        distance_matrix.append(row)
    return distance_matrix

distance_matrix = compute_distance_matrix(all_locations)

# --- STEP 4: Definisikan Data Model untuk OR-Tools ---
def create_data_model():
    data = {}
    data["distance_matrix"] = distance_matrix
    data["demands"] = all_demands
    data["vehicle_capacities"] = [3000, 3000]  # Contoh: 2 kendaraan kapasitas 3000 unit
    data["num_vehicles"] = len(data["vehicle_capacities"])
    data["depot"] = 0
    return data

# --- STEP 5: Setup OR-Tools Routing Model ---
def solve_vrp(data):
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'],
                                           data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]
    
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]
    
    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        routes = []
        for vehicle_id in range(data['num_vehicles']):
            index = routing.Start(vehicle_id)
            route = []
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                route.append(node)
                index = solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))
            routes.append(route)
        return routes
    else:
        return None

# --- STEP 6: Jalankan Model ---
data = create_data_model()
routes = solve_vrp(data)

# --- STEP 7: Tampilkan Output ---
if routes:
    for i, route in enumerate(routes):
        print(f"Rute Kendaraan {i+1}:")
        for node in route:
            if node == 0:
                print("Depot", end=" -> ")
            else:
                print(f"Pelanggan-{node}", end=" -> ")
        print("End\n")
else:
    print("Tidak ditemukan solusi.")
